# Bayesian Networks and Autoregressive Language Models

## Part I: From Probability to Language
**Question 1: Why is this autoregressive decomposition useful for generating text?**
The chain rule decomposes the massive, intractable joint probability of an entire sentence into a sequence of conditional probabilities. This is extremely useful for generation because it perfectly matches the left-to-right sequential nature of language. We can iteratively predict and sample the next word based solely on the prefix we have generated so far, rather than trying to sample a full sequence at once.

## Part II: A Bayesian Network for Text
**Question 2: What independence assumption is being made by this network?**
The network assumes the **First-Order Markov Property**:
$P(X_t | X_1, \dots, X_{t-1}) = P(X_t | X_{t-1})$
It assumes the probability of the next word depends *only* on the immediately preceding word, and is conditionally independent of all other previous words.

## Part IV: Constructing the Conditional Probability Table
**Question 3: Construct the conditional probability distribution**
From the dataset provided:
- `P(cat | the) = 3/6 = 0.5`
- `P(dog | the) = 3/6 = 0.5`
- `P(sat | cat) = 2/3 ≈ 0.66`
- `P(ran | cat) = 1/3 ≈ 0.33`
- `P(sat | dog) = 2/3 ≈ 0.66`
- `P(ran | dog) = 1/3 ≈ 0.33`
- `P(on | sat) = 4/4 = 1.0`
- `P(to | ran) = 2/2 = 1.0`

**Zero-probability transitions:**
`P(sat | the) = 0`, `P(the | cat) = 0`, `P(dog | cat) = 0`, etc. Any word pair not appearing consecutively in the dataset has a probability of exactly 0.



In [ ]:
import random
from collections import defaultdict

sentences = [
    "<START> the cat sat on the mat <END>",
    "<START> the cat sat on the rug <END>",
    "<START> the dog sat on the mat <END>",
    "<START> the dog ran to the park <END>",
    "<START> the cat ran to the park <END>",
    "<START> the dog sat on the rug <END>"
]

tokenized = [s.split() for s in sentences]

class FirstOrderLM:
    def __init__(self, data):
        self.counts = defaultdict(lambda: defaultdict(int))
        self.probs = defaultdict(dict)
        
        # 1. Count transitions
        for sentence in data:
            for i in range(len(sentence) - 1):
                curr_w, next_w = sentence[i], sentence[i+1]
                self.counts[curr_w][next_w] += 1
                
        # 2. Compute probabilities
        for curr_w, next_words in self.counts.items():
            total = sum(next_words.values())
            for next_w, count in next_words.items():
                self.probs[curr_w][next_w] = count / total

    def check_invariants(self):
        print("--- Probability Normalisation Test ---")
        for word in self.probs:
            total = sum(self.probs[word].values())
            print(f"P(* | {word:8}) sum = {total:.2f}")
            assert abs(total - 1.0) < 1e-6

    def generate(self, greedy=False):
        current = "<START>"
        sentence = []
        while current != "<END>":
            if current not in self.probs: break
            
            next_words = list(self.probs[current].keys())
            probabilities = list(self.probs[current].values())
            
            if greedy:
                # Argmax
                best_idx = probabilities.index(max(probabilities))
                current = next_words[best_idx]
            else:
                # Sample
                current = random.choices(next_words, weights=probabilities)[0]
                
            if current != "<END>":
                sentence.append(current)
                
        return " ".join(sentence)

# Initialize and test
lm1 = FirstOrderLM(tokenized)
lm1.check_invariants()

print("\n--- Generating (Sampling) ---")
for _ in range(5): print(lm1.generate(greedy=False))

print("\n--- Generating (Greedy) ---")
for _ in range(5): print(lm1.generate(greedy=True))



## Part VI & VII: Inspecting and Testing
**Question 4: Where are transition counts stored?** In the `self.counts` nested dictionary (`defaultdict(lambda: defaultdict(int))`).
**Question 5: Where is $P(X_t | X_{t-1})$ computed?** Inside the `__init__` function during the loop `self.probs[curr_w][next_w] = count / total`.
**Question 6: How does the program choose the next word?** It samples from the probability distribution using `random.choices(..., weights=probabilities)`. Sampling picks words proportionally to their likelihood, introducing randomness. Always picking the most probable (greedy) makes generation entirely deterministic.
**Question 7: What happens if it encounters an unseen word?** It hits the `if current not in self.probs: break` condition and stops generating early to prevent a `KeyError`.
**Question 8: If a total is 0.87?** It indicates a critical bug in the implementation of the probability calculation (e.g. integer truncation, omitting a token, or floating point cumulative loss), violating the axiom that a valid probability distribution must sum to exactly 1.0.

## Part VIII: Predicting the Next Word
**Question 9: Difference between model and human expectation.**
The model purely reflects the statistical frequency of its specific, tiny training corpus. Humans rely on vast general knowledge, linguistic syntax rules, and semantics. The model might predict "sat" after "dog" purely because it counted it, whereas a human might expect "barked".

## Part X: Deterministic vs Probabilistic Generation
**Question 10: Compare generated sentences.**
Sampling (`Mode B`) produces variation, generating novel recombinations like "the dog ran to the mat". Greedy generation (`Mode A`) is completely deterministic and always collapses to the exact same path ("the cat sat on the mat"), producing zero variation.



In [ ]:
class SecondOrderLM:
    def __init__(self, data):
        self.counts = defaultdict(lambda: defaultdict(int))
        self.probs = defaultdict(dict)
        
        for sentence in data:
            for i in range(len(sentence) - 2):
                context = (sentence[i], sentence[i+1])
                next_w = sentence[i+2]
                self.counts[context][next_w] += 1
                
        for context, next_words in self.counts.items():
            total = sum(next_words.values())
            for next_w, count in next_words.items():
                self.probs[context][next_w] = count / total

    def generate(self):
        # We assume every sequence starts with "<START> the" based on our data
        w1, w2 = "<START>", "the"
        sentence = ["the"]
        
        while True:
            context = (w1, w2)
            if context not in self.probs: break
            
            next_words = list(self.probs[context].keys())
            probabilities = list(self.probs[context].values())
            
            next_w = random.choices(next_words, weights=probabilities)[0]
            
            if next_w == "<END>": break
            sentence.append(next_w)
            w1, w2 = w2, next_w
            
        return " ".join(sentence)

lm2 = SecondOrderLM(tokenized)
print("--- Second-Order Generation ---")
for _ in range(5): print(lm2.generate())



## Part XI & XIII: Comparing the Models

**Question 11: How does the second-order model differ?**
1. **Graph structure**: $X_{t-2} \rightarrow X_t \leftarrow X_{t-1}$. $X_t$ has two parents instead of one.
2. **CPT**: It represents $P(X_t | X_{t-2}, X_{t-1})$. The table is conditioned on a tuple of two words.
3. **Context**: It uses 2 words of context, remembering slightly further back.
4. **Data needed**: It requires exponentially more data. Because the context space grows quadratically ($V^2$), many bi-gram contexts will never appear in a small dataset (Data Sparsity).

**Question 12: Why does increasing context improve prediction but make estimation harder?**
More context provides sharper, more specific information (e.g., distinguishing the context "to the" from "on the"). However, because the size of the conditional probability table grows exponentially with context size ($|V|^n$), a limited dataset will leave almost all table entries empty (zero-probability contexts), making the model extremely brittle and hard to estimate accurately without massive data.

## Part XV: Reflection on the LLM

**Question 13: Why is Approach B preferable?**
Approach B focuses on specifying the mathematical behaviour and invariants (the actual Bayesian model) rather than just asking for code. This ensures the engineer actually understands the representation (CPTs vs Neural Nets), validates the intended behaviour (probability sums to 1), and tests the probabilistic invariants. An LLM acts as an implementation assistant for a rigorously defined scientific model, rather than a black-box oracle.

**Question 14: What did thinking of the LM as a Bayesian Network give you?**
1. **A representation of dependencies**: It clarifies exactly what independence assumptions are made (Markov property).
2. **A factorisation of the joint distribution**: It explains mathematically how we avoid calculating probabilities over full sequences simultaneously.
3. **A way to test implementation**: By treating it as a pure probability distribution, we know that $\sum P(v|w) = 1$, giving us a mathematically rigorous invariant to test our generated code against to find bugs.

